In [1]:
import requests
import pandas as pd 
import numpy as np
import json
import openpyxl
from functools import reduce

In [1]:
symbol = 'Stock_Symbol'

In [3]:
ALPHAVANTAGE_API_KEY="your_api_key_here"

In [4]:
url = "https://www.alphavantage.co/query"

# INCOME STATEMENT API DATA PULL

In [2]:
# this if your function
inc = 'INCOME_STATEMENT'

In [6]:
params = {
    "function":inc,
    "symbol": symbol,
    "apikey":ALPHAVANTAGE_API_KEY
}

In [7]:
response = requests.get(url, params = params)

In [8]:
financial_data_income = response.json()

In [9]:
financial_data_income_readable = json.dumps(financial_data_income, indent = 4)

In [10]:
#print(finance_data_income_readable)

In [11]:
quarterly_income = pd.json_normalize(financial_data_income, record_path = ['quarterlyReports'])

In [12]:
quarterly_income_df = quarterly_income[["fiscalDateEnding","totalRevenue","grossProfit","operatingIncome",
                               "ebitda","netIncome","incomeBeforeTax","incomeTaxExpense"]]

In [13]:
#print(quarterly_income_df)

In [14]:
annual_income = pd.json_normalize(financial_data_income, record_path = ['annualReports'])

In [15]:
annual_income_df = annual_income[["fiscalDateEnding","totalRevenue","grossProfit","operatingIncome","ebitda",
                         "netIncome","incomeBeforeTax","incomeTaxExpense"]]

In [16]:
#print(annual_income_df)

# BALANCE SHEET API DATA PULL 

In [17]:
bal = 'BALANCE_SHEET'

In [18]:
params2 = {
    "function":bal,
    "symbol":symbol,
    "apikey":ALPHAVANTAGE_API_KEY
}

In [19]:
response2 = requests.get(url,params = params2)

In [20]:
financial_data_balance = response2.json()

In [21]:
financial_data_balance_readable = json.dumps(financial_data_income, indent =4)

In [22]:
#print(financial_data_balance_readable)

In [23]:
quarterly_balance = pd.json_normalize(financial_data_balance, record_path =['quarterlyReports'])

In [24]:
quarterly_balance_df = quarterly_balance[["fiscalDateEnding","totalAssets","shortLongTermDebtTotal",
                                          "totalShareholderEquity","cashAndCashEquivalentsAtCarryingValue",
                                          "totalLiabilities"]]

In [25]:
annual_balance = pd.json_normalize(financial_data_balance, record_path = ['annualReports'])

In [26]:
annual_balance_df = annual_balance[["fiscalDateEnding","totalAssets","shortLongTermDebtTotal",
                                          "totalShareholderEquity"]]

# CASH FLOW API DATA PULL

In [27]:
cash = 'CASH_FLOW'

In [28]:
params3 = {
    'function':cash,
    'symbol':symbol,
    'apikey':ALPHAVANTAGE_API_KEY
}

In [29]:
response3 = requests.get(url, params = params3)

In [30]:
financial_data_cash = response3.json()

In [31]:
financia_data_cash_readable = json.dumps(financial_data_cash, indent =4)

In [32]:
#print(financia_data_cash_readable)

In [33]:
quarterly_cash = pd.json_normalize(financial_data_cash, record_path = ['quarterlyReports'])

In [34]:
quarterly_cash_df = quarterly_cash[["fiscalDateEnding","operatingCashflow", "capitalExpenditures"]]

In [35]:
annual_cash = pd.json_normalize(financial_data_cash, record_path =['annualReports'])

In [36]:
annual_cash_df = annual_cash[["fiscalDateEnding","operatingCashflow", "capitalExpenditures"]]

# MERGE QUARTERLY FINANCIAL STATEMENTS 

In [37]:
quarterly_statements_dfs = [quarterly_income_df, 
                              quarterly_balance_df,
                              quarterly_cash_df]

In [38]:
quarterly_statements_merge_df = reduce(lambda left, right: pd.merge(left, right, 
                                                              on ='fiscalDateEnding', 
                                                              how = 'inner')
                                , quarterly_statements_dfs)

In [39]:
#print(quarterly_statements_df)

# MERGE ANNUAL FINANCIAL STATEMENTS

In [40]:
annual_statements_dfs = [annual_income_df,
                      annual_balance_df,
                      annual_cash_df]

In [41]:
annual_statements_merge_df = reduce(lambda left, right: pd.merge(left,right,
                                                                on = 'fiscalDateEnding',
                                                                how = 'inner')
                                   , annual_statements_dfs)

# CLEANING AND FORMATTING QUARTERLY DATAFRAMES 

In [42]:
# convert object 'fiscalDateEnding' to datetime
quarterly_statements_merge_df['fiscalDateEnding'] = pd.to_datetime(quarterly_statements_merge_df['fiscalDateEnding'])


In [43]:
# select all data types columns that are object
quarterly_object_cols = quarterly_statements_merge_df.select_dtypes(include=['object']).columns

In [44]:
# convert all object columns to numeric
for col in quarterly_object_cols:
    quarterly_statements_merge_df[col] = pd.to_numeric(quarterly_statements_merge_df[col], errors = 'coerce')

In [45]:
# fill al "na" records with 0 & final dataframe
quarterly_df = quarterly_statements_merge_df.fillna(0)

In [46]:
#print(quarterly_df)

# CLEANING AND FORMATTING ANNUAL DATAFRAMES 

In [47]:
annual_statements_merge_df['fiscalDateEnding'] = pd.to_datetime(annual_statements_merge_df['fiscalDateEnding'])

In [48]:
annual_object_cols = annual_statements_merge_df.select_dtypes(include=['object']).columns

In [49]:
for col in annual_object_cols:
    annual_statements_merge_df[col] = pd.to_numeric(annual_statements_merge_df[col], errors = 'coerce')

In [50]:
annual_df = annual_statements_merge_df.fillna(0)

In [51]:
#print(annual_df)

# EXPORT QUARTERLY & ANNUAL DATASETS TO AN EXCEL WORKBOOK

In [52]:
quarterly_df.to_excel("PROP_Financials.xlsx", sheet_name = 'quarterly', index = False)

In [53]:
file_path = ("PROP_Financials.xlsx")

In [54]:
with pd.ExcelWriter(file_path, engine= "openpyxl", mode = 'a') as writer:
    annual_df.to_excel(writer, sheet_name = 'annual', index = False)